### Initial setup
<p>
First, a number of core packages will have to be installed. While there is a requirements.txt file that contains the setup that I was able to use, it is worth using a package manager and manually installing the necessary packages with dependencies. These included:
</p>

<ul>
<li><a href='https://pypi.org/project/ultralytics/'>Ultralytics</a>
<li><a href='https://pypi.org/project/roboflow/'>Roboflow</a>
<li><a href='https://pypi.org/project/opencv-python/'>OpenCV</a>
<li><a href='https://pypi.org/project/torch/'>Pytorch</a> and <a href='https://pypi.org/project/torchvision/'>torchvision</a>
<li><a href='https://pypi.org/project/pandas/'>Pandas</a>
</ul>

<p>
The packages used are desgined to be run on an NVIDIA GPU with <a href='https://developer.nvidia.com/cuda-downloads'>CUDA</a>. Depending on the GPU used, pytorch will have to be installed to match the CUDA version of the GPU. This can be checked by running the <code>!nvidia-smi</code> command to input information about the GPU. Then, install torch and torchvision using the CUDA-compatible versions found at <a href='https://pytorch.org/get-started/locally/'>Pytorch's website</a>.
</p>

In [ ]:
# check for valid gpu, along with cuda version for nvidia gpus
!nvidia-smi

In [ ]:
# install dependencies
!pip install -r requirements.txt

In [ ]:
# import libraries
import os
import pandas as pd
import cv2
from roboflow import Roboflow
import ultralytics
from ultralytics import YOLO

If ultralytics, cuda and pytorch have been correctly installed, then this ouput will show all three, with a recognised GPU as well

In [ ]:
# check ultralytics, pytorch and cuda installation
ultralytics.checks()

### Dataset
<p>The next step is to download the dataset from Roboflow. The only required information is you API key, along with the dataset version (found <a href='https://app.roboflow.com/trail-camera/sam-barrett/10'>here</a>) and the dataset format (YOLOv12 for this example). This will then install the dataset into the project directory. The downloaded dataset is automatically split into train, test and validate subsets, with both images and annotations. In addition, there is a <code>.yaml</code> file that can be used to read in the dataset in its entirety
</p>
<p>
The working directory is also found to create a relative filepath for <code>home</code>, which is used throughout the code.
</p>

In [ ]:
# establish home directory for project
home = os.getcwd()
home

This specific block of code is generated by Roboflow during when <a href='https://app.roboflow.com/trail-camera/sam-barrett/10/export'>downloading a dataset</a>, selecting the 'show download code' option.

In [ ]:
# import dataset from roboflow
rf = Roboflow(api_key="")
project = rf.workspace("trail-camera").project("sam-barrett")
version = project.version(10)           # update version number as needed with new datasets
dataset = version.download("yolov12")   # change to desired yolo version if needed
dataset.location

### Model training and validating
<p>
First, the desired YOLO model is loaded for training, with the specific version and size selected based on the input argument. The model is then trained on the Roboflow dataset, with hyperparameters (such as epochs) configurable; a complete list of training parameters can be found <a href='https://docs.ultralytics.com/modes/train/#train-settings'>here</a>

<p>
Additional information on training can be found <a href='https://docs.ultralytics.com/modes/train/'>here</a>.
</p>


In [ ]:
# choose model size and version to train


# different YOLOv12 sizes
model = YOLO("yolo12n.pt")
#model = YOLO("yolo12s.pt")
#model = YOLO("yolo12m.pt")

# different YOLO versions
#model = YOLO("yolo11m.pt")
#model = YOLO("yolo10m.pt")


# train model
results = model.train(data=(f"{dataset.location}/data.yaml"), epochs=100, device=0, plots=True)

In [ ]:
# select best model from training for validation and inference
run = ""

model = YOLO(f"{home}/runs/detect/train{run}/weights/best.pt")

In [ ]:
# validate model
metrics = model.val(data=(f"{dataset.location}/data.yaml"))

### https://docs.ultralytics.com/modes/predict/#inference-arguments

In [ ]:
# for every video in source directory, run detection and save results to csv
for video in os.listdir(f"{home}/videos"):
    if video.endswith(".mp4"):
        # set source to video filepath
        source = (f"{home}/videos/{video}")

        # capture video for detection
        cap = cv2.VideoCapture(source)

        # empty list to store box data
        data = []
        # frame counter to store for each box
        frame_count = 0

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            frame_count += 1

            # track objects in the current frame
            results = model.track(frame, persist=True)
            #results = model.track(frame, persist=True, save=True) # generates annotated video
            #results = model.track(frame, persist=True,classes=[0,2,3,4,5,6], save=True) # omit class 1 (feeder) from tracking

            # for each box found in each frame, store the box data
            for result in results:
                if result.boxes is not None and len(result.boxes) > 0:
                    for i in range(len(result.boxes)):
                        data.append({
                            # if not none else none handles missing values, while int() and float() convert from tensor to numbers
                            'frame': frame_count,
                            'id': int(result.boxes.id[i]) if result.boxes.id is not None else None,
                            'class': model.names[int(result.boxes.cls[i]) if result.boxes.cls is not None else None], # sets class to nominal name
                            'confidence': float(result.boxes.conf[i]) if result.boxes.conf is not None else None,
                            'x1': int(result.boxes.xyxy[i][0]) if result.boxes.xyxy[i][0] is not None else None,
                            'y1': int(result.boxes.xyxy[i][1]) if result.boxes.xyxy[i][1] is not None else None,
                            'x2': int(result.boxes.xyxy[i][2]) if result.boxes.xyxy[i][2] is not None else None,
                            'y2': int(result.boxes.xyxy[i][3]) if result.boxes.xyxy[i][3] is not None else None
                        })

        # create dataframe from box data list
        df = pd.DataFrame(data)
        # save results to csv
        df.to_csv(f"{home}/results/{video}_results.csv", index=False, header=True)
    else:
        print(f"{video} is not a valid video file (.mp4)")

In [ ]:
# class name values
model.names